In [0]:
import requests
import json
from pyspark.sql import Row
from pyspark.sql.functions import explode
import time

def processed_data(batch_df,batch_id):
    key=dbutils.secrets.get(scope="stream-project-secret",key="weather_api_key")
    base_url=" http://api.weatherapi.com/v1"
    current_weather_url=f"{base_url}/current.json"
    location='Kolkata'
    param={
        "key": key,
        "q": location,
    }
    response=requests.get(current_weather_url,params=param)
    # data=response.json()
    current_weather=response.json()

    location_data=current_weather.get('location',{})
    current_data=current_weather.get('current',{})
    flattened_data={
        'name': location_data.get('name'),
        'region': location_data.get('region'),
        'country': location_data.get('country'),
        'lat': location_data.get('lat'),
        'lon': location_data.get('lon'),
        'localtime_epoch':location_data.get('localtime_epoch'),
        'localtime':location_data.get('localtime'),
        'last_updated':current_data.get('last_updated'),
        'temp_c':current_data.get('temp_c'),
        'temp_f':current_data.get('temp_f'),
        'wind_mph':current_data.get('wind_mph'),
        'humidity':current_data.get('humidity'),
        'feelslike_c':current_data.get('feelslike_c')
    }
    df=spark.createDataFrame([flattened_data])
    df.write.format("delta").partitionBy('last_updated').mode("append").option('overWriteSchema','true').saveAsTable("hive_metastore.mytabs.weather_streaming_table")
    # df.display()

In [0]:
def main():
    streamingdf = spark.readStream.format("rate").option("rowsPerSecond", 1).load()
    query = (
        streamingdf.writeStream.foreachBatch(processed_data)
        .trigger(processingTime="60 seconds")
        .option("checkpointLocation", "/tmp/spark/checkpoints")
        .start()
    )
    time.sleep(60)
    query.stop()


if __name__ == "__main__":
    main()

In [0]:
##stream-project-secret weather_api_key
# dbutils.secrets.listScopes()

In [0]:
class DA:
    paths = {"events": "dbfs:/user/hive/warehouse/mytabs.db/weather_streaming_table"}

In [0]:
streaming_df_new = spark.readStream.load(DA.paths["events"]).select("*")
display(streaming_df_new, streamName="display_weather_details")

In [0]:
checkpoint_path = "/tmp/spark/checkpoints_new"
weather_query = (
    streaming_df_new.writeStream.outputMode("append")
    .option('overWriteSchema','true')
    .format("delta")
    .queryName("weather_details_filtered")
    .trigger(processingTime="1 second")
    .option("checkpointLocation", checkpoint_path)
    .toTable("hive_metastore.mytabs.weather_streaming_table_new")
)

In [0]:
%sql
select * from hive_metastore.mytabs.weather_streaming_table_new;